# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

## Step 0: Install dependencies

In [33]:

%pip install -q sentence-transformers faiss-cpu pypdf pandas numpy requests

Note: you may need to restart the kernel to use updated packages.


## Step 1: Configuration and the PDF

In [34]:
# ---- USER CONFIG ----
PDF_PATH = "icc_test_playing_conditions.pdf"  # <- your PDF file name (same folder as this notebook)

# Models (downloaded automatically from Hugging Face on first use)
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# LLM: "gemini" | "openai" | "anthropic" | "ollama"
LLM_PROVIDER = "ollama"
GEMINI_MODEL = "gemini-2.5-flash"
OPENAI_MODEL = "gpt-4o-mini"
ANTHROPIC_MODEL = "claude-sonnet-5-5"
OLLAMA_MODEL = "llama3.1"  # must be pulled first: `ollama pull llama3.1` (or llama3.2 for a lighter model)

# Chunking
CHUNK_WORDS = 180
OVERLAP_WORDS = 40
TOP_K_RETRIEVE = 10
TOP_K_FINAL = 3

In [35]:
# OPTIONAL: trim the PDF to a page range (e.g. first 80 pages). Set TRIM = True to enable.
TRIM, FIRST_PAGE, LAST_PAGE = False, 1, 80  # 1-based, inclusive

if TRIM:
    from pypdf import PdfReader, PdfWriter

    r, w = PdfReader(PDF_PATH), PdfWriter()
    for p in range(FIRST_PAGE - 1, min(LAST_PAGE, len(r.pages))):
        w.add_page(r.pages[p])
    trimmed = PDF_PATH.replace(".pdf", f"_p{FIRST_PAGE}-{LAST_PAGE}.pdf")
    w.write(open(trimmed, "wb"))
    PDF_PATH = trimmed
    print("Trimmed ->", PDF_PATH, f"({len(w.pages)} pages)")
else:
    print("Using full PDF:", PDF_PATH)

Using full PDF: icc_test_playing_conditions.pdf


## Step 2: LLM setup (Ollama by default)

**Ollama (default, local):** no key needed. Make sure it is running (`curl http://localhost:11434` prints *Ollama is running*) and a model is pulled (`ollama pull llama3.1` or `ollama pull llama3.2`). The check cell below verifies this.

In [36]:
import os, getpass

KEY_NAMES = {
    "gemini": "GEMINI_API_KEY",
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}


def load_api_key(provider):
    if provider == "ollama":
        return None
    name = KEY_NAMES[provider]
    if os.environ.get(name):
        return os.environ[name]
    try:  # Colab Secrets
        from google.colab import userdata

        os.environ[name] = userdata.get(name)
        return os.environ[name]
    except Exception:
        pass
    os.environ[name] = getpass.getpass(
        f"Enter {name}: "
    )  # fallback prompt (hidden input)
    return os.environ[name]


load_api_key(LLM_PROVIDER)
print("LLM provider:", LLM_PROVIDER)

LLM provider: ollama


In [37]:
# Ollama connectivity check + automatic model selection
if LLM_PROVIDER == "ollama":
    import requests

    try:
        tags = requests.get("http://localhost:11434/api/tags", timeout=5).json()
        all_models = [m["name"] for m in tags.get("models", [])]
        print("Ollama is running. Installed models:", all_models)
        installed = [
            m for m in all_models if "embed" not in m.lower()
        ]  # embedding models cannot generate text
        if not installed:
            raise RuntimeError("No chat model installed. Run: ollama pull llama3.1")
        if not any(
            m == OLLAMA_MODEL or m.startswith(OLLAMA_MODEL + ":") for m in installed
        ):
            print(f"'{OLLAMA_MODEL}' not found; switching to '{installed[0]}'")
            OLLAMA_MODEL = installed[0]
        print("Using model:", OLLAMA_MODEL)
    except requests.exceptions.ConnectionError:
        raise RuntimeError(
            "Ollama is not running. Open the Ollama app or run `ollama serve` in Terminal."
        )

Ollama is running. Installed models: ['llama3.1:latest', 'nomic-embed-text:latest', 'granite3.1-dense:8b']
Using model: llama3.1


## Step 3: Download and load the encoders

- **Bi-Encoder:** `all-MiniLM-L6-v2` (384-dim dense embeddings).
- **Cross-Encoder:** `cross-encoder/ms-marco-MiniLM-L-6-v2`.

Both come from the public Hugging Face Hub and are cached after the first download (~90 MB and ~90 MB). If you hit rate limits or want a token, optionally create a free token at https://huggingface.co/settings/tokens and set `HF_TOKEN` the same way as above.

*Offline / manual alternative:* `huggingface-cli download cross-encoder/ms-marco-MiniLM-L-6-v2 --local-dir ./ce` and then set `CROSS_ENCODER_NAME = "./ce"` (same for the bi-encoder).

In [38]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=device)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=device)
print("Loaded on:", device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loaded on: mps


## Step 4: Load the PDF and chunk it (shared corpus for both pipelines)


In [39]:
import re


def avg_word_len(text):
    w = text.split()
    return sum(map(len, w)) / max(len(w), 1)


def extract_pages(path):
    try:
        import fitz  # PyMuPDF (cleanest extractor, if installed)

        return [p.get_text("text") for p in fitz.open(path)]
    except ImportError:
        from pypdf import PdfReader

        return [p.extract_text() or "" for p in PdfReader(path).pages]


def fix_letter_spacing(text):
    # "t h e  f a c t" -> "the fact": letters split by 1 space, words by 2+ spaces
    if avg_word_len(text) >= 3:
        return text
    return " ".join(w.replace(" ", "") for w in re.split(r"\s{2,}", text))


page_texts = [
    re.sub(r"\s+", " ", fix_letter_spacing(t)).strip() for t in extract_pages(PDF_PATH)
]
print(
    "avg word length:", round(avg_word_len(" ".join(page_texts)), 2), "(should be ~4-6)"
)

words_with_page = [
    (w, pno) for pno, t in enumerate(page_texts, start=1) for w in t.split(" ") if w
]
chunks, i = [], 0
step = CHUNK_WORDS - OVERLAP_WORDS
while i < len(words_with_page):
    window = words_with_page[i : i + CHUNK_WORDS]
    chunks.append(
        {
            "id": len(chunks),
            "page": window[0][1],
            "text": " ".join(w for w, _ in window),
        }
    )
    i += step

print(f"{len(page_texts)} pages, {len(words_with_page)} words -> {len(chunks)} chunks")
print("\nSample chunk:", chunks[min(5, len(chunks) - 1)]["text"][:300], "...")

avg word length: 5.02 (should be ~4-6)
103 pages, 45594 words -> 326 chunks

Sample chunk: not within the playing area described in clause1.2.5.2 above (for example, the player is not permitted to enter the on-field ‘dugout’). 1.2.7Concussion Replacement 1.2.7.1If a player sustains a concussion or suspected concussion as a result of a head or neck injury during the course of the relevant  ...


## Step 5: Build the vector index (Bi-Encoder + FAISS inner product on normalised vectors = cosine)

In [40]:
import numpy as np, faiss

chunk_texts = [c["text"] for c in chunks]
chunk_emb = bi_encoder.encode(
    chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True
).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print("Indexed vectors:", index.ntotal)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Indexed vectors: 326


## Step 6: Pipelines 1 and 2

In [41]:
import time, re, textwrap, difflib
import numpy as np
import pandas as pd

STOPWORDS = {
    "what",
    "when",
    "does",
    "which",
    "that",
    "this",
    "with",
    "from",
    "after",
    "before",
    "have",
    "there",
    "their",
    "about",
}

# OPTIONAL answer key: lets the notebook judge the results automatically.
#   "clause_text"  = a short piece of text that appears in the chunk holding the right rule (plain text, not a regex)
#   "answer_words" = words that a correct answer must contain
# Fill this in for your failure queries in Step 9 (use find_chunks("some text") to locate the right clause).
ANSWER_KEY = {
    "What are the rules about the maximum size of the bat?": {
        "clause_text": "52%",
        "answer_words": ["38 inches", "52%"],
    },
    "What is the minimum number of overs that must be bowled on the last day of a Test match?": {
        "clause_text": None,
        "answer_words": ["75"],
    },
}

ANSWER_KEY.update(
    {
        "How much extra time can be used to complete the minimum overs?": {
            "clause_text": "permitted overtime",
            "answer_words": ["30 minutes"],
        },
        "Can a team nominate a substitute fielder after the toss?": {
            "clause_text": "subsequent additions",
            "answer_words": ["exceptional circumstances"],
        },
        "Can a declaration or follow-on decision be withdrawn after it has been notified?": {
            "clause_text": "15.3Notification",
            "answer_words": ["cannot be changed"],
        },
        "Are unbowled overs carried forward to the next day?": {
            "clause_text": "shall not be made up on any subsequent day",
            "answer_words": ["not be made up"],
        },
        "What is the penalty for running short deliberately?": {
            "clause_text": "award five (5) Penalty runs",
            "answer_words": ["five (5) penalty runs"],
        },
        "Can a captain declare in the final innings of the match?": {
            "clause_text": "With the exception of the final innings",
            "answer_words": ["cannot"],
        },
    }
)


# ------------------------------------------------------------------ the two pipelines
def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Pipeline 1 retrieval: dense top-k by cosine similarity. Returns a list of (chunk_id, cosine)."""
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]


def rerank(query, candidates, k=TOP_K_FINAL):
    """Pipeline 2: score every (query, chunk) pair with the cross-encoder, then sort by that score."""
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    out = []
    for rank_before, ((cid, cosine), ce) in enumerate(
        zip(candidates, ce_scores), start=1
    ):
        out.append(
            {"id": cid, "cosine": cosine, "ce": float(ce), "orig_rank": rank_before}
        )
    out.sort(key=lambda d: d["ce"], reverse=True)
    for rank_after, d in enumerate(out, start=1):
        d["new_rank"] = rank_after
    return out[:k], out  # (top-k after re-ranking, all 10 after re-ranking)


# ------------------------------------------------------------------ identical prompt for both answers
def build_prompt(query, chunk_ids):
    ctx = "\n\n".join(
        f"[Chunk {n} | page {chunks[c]['page']}]\n{chunks[c]['text']}"
        for n, c in enumerate(chunk_ids, start=1)
    )
    return (
        "You are a careful assistant answering questions about a document.\n"
        "Use ONLY the context below. If the context does not contain the answer, say so. "
        "Mention any exceptions or restrictions that appear in the context.\n\n"
        f"CONTEXT:\n{ctx}\n\nQUESTION: {query}\n\nANSWER:"
    )


def generate(prompt):
    if LLM_PROVIDER == "gemini":
        from google import genai

        client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        return client.models.generate_content(
            model=GEMINI_MODEL, contents=prompt
        ).text.strip()
    if LLM_PROVIDER == "openai":
        from openai import OpenAI

        r = OpenAI().chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
        )
        return r.choices[0].message.content.strip()
    if LLM_PROVIDER == "anthropic":
        import anthropic

        r = anthropic.Anthropic().messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=600,
            messages=[{"role": "user", "content": prompt}],
        )
        return r.content[0].text.strip()
    if LLM_PROVIDER == "ollama":
        import requests

        r = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0, "num_ctx": 4096},
            },
            timeout=300,
        )
        if not r.ok:
            raise RuntimeError(
                f"Ollama {r.status_code} for model '{OLLAMA_MODEL}': {r.text}"
            )
        return r.json()["response"].strip()
    raise ValueError("Unknown LLM_PROVIDER")


def run_both(query):
    """Run Pipeline 1 and Pipeline 2 for one query. Returns everything we need, including timings."""
    t0 = time.perf_counter()
    cands = baseline_retrieve(query)  # top 10 from the bi-encoder
    t1 = time.perf_counter()
    base_top3 = [
        cid for cid, _ in cands[:TOP_K_FINAL]
    ]  # Pipeline 1: just take the first 3
    rr_top3, rr_all = rerank(query, cands)  # Pipeline 2: cross-encoder re-sort
    t2 = time.perf_counter()
    ans1 = generate(build_prompt(query, base_top3))
    t3 = time.perf_counter()
    ans2 = generate(build_prompt(query, [d["id"] for d in rr_top3]))
    t4 = time.perf_counter()
    return {
        "query": query,
        "base_top3": base_top3,
        "rr_top3": rr_top3,
        "rr_all": rr_all,
        "ans1": ans1,
        "ans2": ans2,
        "t_baseline": t1 - t0,
        "t_rerank": t2 - t1,
        "t_llm1": t3 - t2,
        "t_llm2": t4 - t3,
    }


# ------------------------------------------------------------------ helpers for readable printing
def short_text(text, query, length=300):
    """A short piece of a chunk, starting near the first useful query word found in it."""
    text = " ".join(text.split())
    if len(text) <= length:
        return text
    start = 0
    for word in re.findall(r"[a-z0-9%]+", query.lower()):
        if len(word) > 3 and word not in STOPWORDS and word in text.lower():
            start = max(0, text.lower().find(word) - 60)
            break
    return ("..." if start > 0 else "") + text[start : start + length] + "..."


def find_chunks(some_text):
    """Print every chunk that contains some_text (not case sensitive) and return their ids."""
    ids = []
    for c in chunks:
        pos = c["text"].lower().find(some_text.lower())
        if pos >= 0:
            ids.append(c["id"])
            print(
                f"chunk {c['id']} (page {c['page']}): ...{c['text'][max(0, pos - 60):pos + 150]}..."
            )
    print(f"\n{len(ids)} chunk(s) found.")
    return ids


def right_chunks(query):
    """Ids of the chunks that hold the right clause for this query (from ANSWER_KEY), or []."""
    clause = ANSWER_KEY.get(query, {}).get("clause_text")
    if not clause:
        return []
    return [c["id"] for c in chunks if clause.lower() in c["text"].lower()]


def has_words(answer, query):
    """Does the answer contain all the words listed in ANSWER_KEY? Returns Yes / No / not set."""
    words = ANSWER_KEY.get(query, {}).get("answer_words")
    if not words:
        return "not set"
    return "Yes" if all(w.lower() in answer.lower() for w in words) else "No"


def similarity(a, b):
    """How similar are two answers, 0-100 percent."""
    return round(
        100
        * difflib.SequenceMatcher(None, a.lower().split(), b.lower().split()).ratio()
    )


def entered_and_left(res):
    """Which chunks entered the top 3 after re-ranking, and which left it."""
    before = set(res["base_top3"])
    after = {d["id"] for d in res["rr_top3"]}
    entered = [d for d in res["rr_all"] if d["id"] in after and d["id"] not in before]
    left = [d for d in res["rr_all"] if d["id"] in before and d["id"] not in after]
    return entered, left


def result_sentence(res):
    """One plain sentence describing what re-ranking did for this query."""
    entered, left = entered_and_left(res)
    right = right_chunks(res["query"])
    if right:
        top10_ids = [d["id"] for d in res["rr_all"]]
        in_before = any(c in res["base_top3"] for c in right)
        in_after = any(c in [d["id"] for d in res["rr_top3"]] for c in right)
        if not any(c in top10_ids for c in right):
            return (
                "The right clause is not in the top 10, so neither pipeline found it."
            )
        if in_after and not in_before:
            return "RE-RANKING HELPED: the right clause was outside the baseline top 3 and is now inside it."
        if in_before and in_after:
            return "No difference: both pipelines found the right clause."
        if in_before and not in_after:
            return "RE-RANKING HURT: the right clause was in the baseline top 3 but was dropped."
        return "The right clause is in the top 10 but in neither top 3."
    if entered:
        names = ", ".join(
            f"chunk {d['id']} (rank {d['orig_rank']} -> {d['new_rank']})"
            for d in entered
        )
        return f"These chunks entered the top 3: {names}. Read them and the answers to decide if this is a real failure."
    return "The same chunks are in the top 3 for both pipelines."


# ------------------------------------------------------------------ the main printing function
def rank_table(res):
    """A plain table of the top 10 chunks: rank before vs rank after re-ranking."""
    before = set(res["base_top3"])
    after = {d["id"] for d in res["rr_top3"]}
    right = right_chunks(res["query"])
    rows = []
    for d in res["rr_all"]:
        change = d["orig_rank"] - d["new_rank"]  # positive = moved up
        note = ""
        if d["id"] in after and d["id"] not in before:
            note = "ENTERED top 3"
        elif d["id"] in before and d["id"] not in after:
            note = "LEFT top 3"
        if d["id"] in right:
            note += "  <- right clause"
        rows.append(
            {
                "Rank_after": d["new_rank"],
                "Rank_before": d["orig_rank"],
                "Change": f"+{change}" if change > 0 else str(change),
                "Chunk": d["id"],
                "Page": chunks[d["id"]]["page"],
                "Cosine": round(d["cosine"], 3),
                "CE_score": round(d["ce"], 2),
                "Note": note.strip(),
            }
        )
    return pd.DataFrame(rows)


def report(res, full_text=False):
    """Print everything about one query in 5 short parts."""
    query = res["query"]
    line = "=" * 90
    print(line)
    print("QUERY:", query)
    print(line)

    print("\nPART 1 - RANKING OF THE TOP 10 CHUNKS")
    print("  Rank_before = position given by the bi-encoder (cosine similarity)")
    print(
        "  Rank_after  = position after the cross-encoder re-sorts the same 10 chunks"
    )
    print("  Change      = places moved (+ means moved up, - means moved down)")
    print(
        "  Cosine      = bi-encoder similarity | CE_score = cross-encoder score (higher = more relevant)\n"
    )
    print(rank_table(res).to_string(index=False))

    print("\nPART 2 - CHUNKS THAT ENTERED OR LEFT THE TOP 3")
    entered, left = entered_and_left(res)
    if not entered and not left:
        print("  None. The same 3 chunks are used by both pipelines.")
    for d in entered:
        print(
            f"\n  Chunk {d['id']} (page {chunks[d['id']]['page']}) ENTERED the top 3: rank {d['orig_rank']} -> {d['new_rank']}"
        )
        body = (
            chunks[d["id"]]["text"]
            if full_text
            else short_text(chunks[d["id"]]["text"], query)
        )
        print(textwrap.fill(body, 88, initial_indent="    ", subsequent_indent="    "))
    for d in left:
        print(
            f"\n  Chunk {d['id']} (page {chunks[d['id']]['page']}) LEFT the top 3: rank {d['orig_rank']} -> {d['new_rank']}"
        )
        body = (
            chunks[d["id"]]["text"]
            if full_text
            else short_text(chunks[d["id"]]["text"], query)
        )
        print(textwrap.fill(body, 88, initial_indent="    ", subsequent_indent="    "))

    print("\nPART 3 - THE TWO ANSWERS (same prompt, different chunks)")
    for title, key, ids in [
        ("ANSWER 1 (baseline, top 3 chunks)", "ans1", res["base_top3"]),
        (
            "ANSWER 2 (re-ranked, top 3 chunks)",
            "ans2",
            [d["id"] for d in res["rr_top3"]],
        ),
    ]:
        print(f"\n  {title} - chunks used: {ids}")
        print(f"  Contains the expected words? {has_words(res[key], query)}")
        for paragraph in res[key].strip().splitlines():
            print(
                textwrap.fill(
                    paragraph, 88, initial_indent="    ", subsequent_indent="    "
                )
                if paragraph.strip()
                else ""
            )
    print(f"\n  Similarity of the two answers: {similarity(res['ans1'], res['ans2'])}%")

    print("\nPART 4 - RESULT")
    print("  " + result_sentence(res))

    print("\nPART 5 - TIME TAKEN")
    print(f"  Retrieval (bi-encoder): {res['t_baseline']*1000:.1f} ms")
    print(f"  Re-ranking (cross-encoder, extra): {res['t_rerank']*1000:.1f} ms")
    print(
        f"  LLM answer 1: {res['t_llm1']:.1f} s | LLM answer 2: {res['t_llm2']:.1f} s\n"
    )

### How to read the output of each query
Every query prints 5 short parts:

1. **Ranking of the top 10 chunks** - `Rank_before` is the bi-encoder position (cosine similarity), `Rank_after` is the position after the cross-encoder re-sorts the same 10 chunks. `Change` shows how many places a chunk moved (`+7` = moved up 7). `Cosine` and `CE_score` are the two scores (higher = more relevant). The `Note` column marks chunks that **ENTERED** or **LEFT** the top 3.
2. **Chunks that entered or left the top 3** - a short piece of text from each one, so you can see whether the clause that answers the question was missing from Answer 1.
3. **The two answers** - Answer 1 (baseline top 3) and Answer 2 (re-ranked top 3), written by the LLM from an identical prompt.
4. **Result** - one sentence saying what re-ranking did. It is automatic only if you fill in `ANSWER_KEY` (optional); otherwise it tells you which chunks to check by hand.
5. **Time taken** - retrieval, re-ranking and LLM time for this query.

## Step 7: Smoke test

First check that the LLM answers, then run one complete query. The section above explains how to read the printed report.

In [42]:
print(generate("Reply with the single word: ready"))  # LLM connectivity test

Ready


In [43]:
demo = run_both(
    "What are the rules about the maximum size of the bat?"
)  # replace with your own question
report(demo)

QUERY: What are the rules about the maximum size of the bat?

PART 1 - RANKING OF THE TOP 10 CHUNKS
  Rank_before = position given by the bi-encoder (cosine similarity)
  Rank_after  = position after the cross-encoder re-sorts the same 10 chunks
  Change      = places moved (+ means moved up, - means moved down)
  Cosine      = bi-encoder similarity | CE_score = cross-encoder score (higher = more relevant)

 Rank_after  Rank_before Change  Chunk  Page  Cosine  CE_score            Note
          1            1      0     32    12   0.707      5.36 <- right clause
          2            4     +2    277    83   0.529      2.39   ENTERED top 3
          3            3      0    276    82   0.566      0.87                
          4            2     -2     31    12   0.593     -0.54      LEFT top 3
          5            8     +3    115    36   0.460     -6.16                
          6            6      0     26    11   0.503     -6.19                
          7            5     -2     

## Step 8: Find failure queries automatically (optional helper)

Write 20-40 candidate questions about your document (include the four categories below), then this cell scans them **without calling the LLM** and flags every query where the Cross-Encoder promotes a chunk from baseline ranks 4-10 into the Top 3. Pick your best examples from the flagged list.

In [44]:
CANDIDATE_QUERIES = [
    # ---- Exceptions / corner cases ----
    "What is the minimum number of overs that must be bowled on the last day of a Test match?",
    "Can the minimum overs target for a day be reduced when time is lost?",
    "What lead is needed to enforce the follow-on in a match scheduled for fewer than five days?",
    "What happens to the follow-on option if the first day of the match is lost?",
    "When can the fielding captain claim a new ball?",
    "In what circumstances can a concussion replacement be someone not on the nominated list?",
    "Can a team nominate a substitute fielder after the toss?",
    "What happens to the playing time lost to bad light or rain?",
    # ---- Negation / constraints ----
    "Can a captain declare in the final innings of the match?",
    "Can a declaration or follow-on decision be withdrawn after it has been notified?",
    "Are unbowled overs carried forward to the next day?",
    "Is wet ground on its own a reason to suspend play?",
    "Which types of bat are not permitted in a Test match?",
    "Can a player who was not nominated before the toss act as a substitute fielder?",
    "Are runs scored if a batter deliberately runs short?",
    "Can the follow-on be declined by the captain after being notified?",
    # ---- Keyword traps (words overlap an unrelated clause) ----
    "How many penalty runs are given for a no-ball?",  # 'ball' overlaps the ball-condition clauses
    "What is the maximum length of the bat handle?",  # 'length' overlaps pitch/boundary length
    "How much extra time can be used to complete the minimum overs?",  # answer uses the word 'overtime'
    "Who decides whether it is too dark to continue playing?",  # answer uses 'light' and 'umpires'
    "What happens when a player suffers a blow to the head?",  # answer uses 'concussion replacement'
    # ---- Other (facts likely buried in a long clause) ----
    "How many substitute fielders can a team name before the match?",
    "How long after the injury can a concussion replacement be requested?",
    "What is the maximum boundary length?",
    "What is the penalty for running short deliberately?",
    "After how many overs can a new ball be requested?",
]

rows = []
for q in CANDIDATE_QUERIES:
    cands = baseline_retrieve(q)
    top3, all10 = rerank(q, cands)
    entered = [
        d
        for d in all10
        if d["new_rank"] <= TOP_K_FINAL and d["orig_rank"] > TOP_K_FINAL
    ]
    rows.append(
        {
            "Query": q,
            "Entered_top3 (chunk: rank before -> after)": ", ".join(
                f"{d['id']}: {d['orig_rank']}->{d['new_rank']}" for d in entered
            )
            or "none",
            "How_many": len(entered),
            "Biggest_jump": max(
                [d["orig_rank"] - d["new_rank"] for d in entered], default=0
            ),
        }
    )

scan = (
    pd.DataFrame(rows)
    .sort_values(["How_many", "Biggest_jump"], ascending=False)
    .reset_index(drop=True)
)
print(
    "Queries sorted by how many chunks the cross-encoder pulled into the top 3 (most first)."
)
print(
    "Entering the top 3 is only a real failure if the new chunk contains the answer - check this in Step 9.\n"
)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
print(scan.to_string(index=False))

Queries sorted by how many chunks the cross-encoder pulled into the top 3 (most first).
Entering the top 3 is only a real failure if the new chunk contains the answer - check this in Step 9.

                                                                                      Query Entered_top3 (chunk: rank before -> after)  How_many  Biggest_jump
                                         Is wet ground on its own a reason to suspend play?                        14: 9->1, 19: 10->3         2             8
                                   Can a captain declare in the final innings of the match?                         56: 9->2, 90: 7->3         2             7
                                    Who decides whether it is too dark to continue playing?                         14: 8->1, 67: 6->3         2             7
           Can a declaration or follow-on decision be withdrawn after it has been notified?                        91: 8->2, 297: 7->3         2             6
             

## Step 8b: Run ALL candidate questions and save a CSV (to choose the failure queries)

This cell runs every question in `CANDIDATE_QUERIES` through **both pipelines** and writes **`candidate_questions_and_responses.csv`**. It takes about 2 LLM calls per question (roughly 4-6 minutes for 26 questions with `llama3.1`). The CSV is saved after every question, so partial results are kept if you stop early.

**How to pick `FAILURE_QUERIES` from the CSV**
1. Filter the column `Failure_candidate` for rows starting with **YES** (the top 3 chunks changed *and* the answers differ).
2. Read `Entered_chunk_text`: does the chunk that entered the top 3 actually contain the answer?
3. Compare `Answer_1_baseline` with `Answer_2_reranked`: is Answer 1 wrong, incomplete, or missing an exception that Answer 2 has?
4. Keep the queries where both are true, at least one per category (exception, negation, keyword trap, other).

In [45]:
CSV_NAME = "candidate_questions_and_responses.csv"

# Category of each question (same order and counts as the groups in CANDIDATE_QUERIES above)
CATEGORY_COUNTS = [
    ("Exception / corner case", 8),
    ("Negation / constraint", 8),
    ("Keyword trap", 5),
    ("Other (buried fact)", 5),
]
categories = []
for name, count in CATEGORY_COUNTS:
    categories += [name] * count
if len(categories) != len(CANDIDATE_QUERIES):  # list was edited: fall back to one label
    categories = ["Candidate"] * len(CANDIDATE_QUERIES)


def chunk_notes(chunk_list, query):
    """Chunk id, page and a short excerpt for each chunk in the list."""
    return (
        " || ".join(
            f"chunk {d['id']} (page {chunks[d['id']]['page']}): {short_text(chunks[d['id']]['text'], query, 350)}"
            for d in chunk_list
        )
        or "none"
    )


candidate_rows = []
for n, (category, q) in enumerate(zip(categories, CANDIDATE_QUERIES), start=1):
    print(f"[{n}/{len(CANDIDATE_QUERIES)}] {q}")
    res = run_both(q)
    entered, left = entered_and_left(res)
    sim = similarity(res["ans1"], res["ans2"])
    if entered and sim < 95:
        flag = "YES - top-3 chunks changed and the answers differ"
    elif entered:
        flag = "MAYBE - top-3 chunks changed but the answers look alike"
    else:
        flag = "NO - same chunks in the top 3"
    candidate_rows.append(
        {
            "No": n,
            "Category": category,
            "Question": q,
            "Failure_candidate": flag,
            "Baseline_top3_chunks": ", ".join(
                f"{c} (page {chunks[c]['page']})" for c in res["base_top3"]
            ),
            "Reranked_top3_chunks": ", ".join(
                f"{d['id']} (page {chunks[d['id']]['page']})" for d in res["rr_top3"]
            ),
            "Entered_top3 (chunk: rank before -> after)": ", ".join(
                f"{d['id']}: {d['orig_rank']}->{d['new_rank']}" for d in entered
            )
            or "none",
            "Left_top3 (chunk: rank before -> after)": ", ".join(
                f"{d['id']}: {d['orig_rank']}->{d['new_rank']}" for d in left
            )
            or "none",
            "Entered_chunk_text": chunk_notes(entered, q),
            "Left_chunk_text": chunk_notes(left, q),
            "Answer_1_baseline": res["ans1"],
            "Answer_2_reranked": res["ans2"],
            "Answer_similarity_pct": sim,
            "Best_chunk_before": res["base_top3"][0],
            "Best_chunk_after": res["rr_top3"][0]["id"],
            "Rerank_extra_ms": round(res["t_rerank"] * 1000, 1),
        }
    )
    pd.DataFrame(candidate_rows).to_csv(
        CSV_NAME, index=False, encoding="utf-8-sig"
    )  # saved after every question

cand_df = pd.DataFrame(candidate_rows)
print(f"\nSaved {len(cand_df)} rows to {CSV_NAME}\n")
print(
    cand_df[
        ["No", "Category", "Failure_candidate", "Answer_similarity_pct", "Question"]
    ].to_string(index=False)
)
print(
    "\nQuestions flagged YES:",
    int(cand_df["Failure_candidate"].str.startswith("YES").sum()),
    "| MAYBE:",
    int(cand_df["Failure_candidate"].str.startswith("MAYBE").sum()),
    "| NO:",
    int(cand_df["Failure_candidate"].str.startswith("NO").sum()),
)

[1/26] What is the minimum number of overs that must be bowled on the last day of a Test match?
[2/26] Can the minimum overs target for a day be reduced when time is lost?
[3/26] What lead is needed to enforce the follow-on in a match scheduled for fewer than five days?
[4/26] What happens to the follow-on option if the first day of the match is lost?
[5/26] When can the fielding captain claim a new ball?
[6/26] In what circumstances can a concussion replacement be someone not on the nominated list?
[7/26] Can a team nominate a substitute fielder after the toss?
[8/26] What happens to the playing time lost to bad light or rain?
[9/26] Can a captain declare in the final innings of the match?
[10/26] Can a declaration or follow-on decision be withdrawn after it has been notified?
[11/26] Are unbowled overs carried forward to the next day?
[12/26] Is wet ground on its own a reason to suspend play?
[13/26] Which types of bat are not permitted in a Test match?
[14/26] Can a player who was n

## Step 9: Failure-mode analysis (stress tests)

Each failure query prints 5 parts: **1) ranking table, 2) chunks that entered or left the top 3, 3) the two answers, 4) result, 5) time taken.** A summary and ready-to-use rank-shift sentences follow in the next cell.

For every query, read parts 2 and 3 and note in the write-up table whether Answer 1 hallucinated, omitted a clause, or misstated a rule.

In [46]:
FAILURE_QUERIES = {
    "Keyword trap / distractor": "How much extra time can be used to complete the minimum overs?",
    "Exception / corner case": "Can a team nominate a substitute fielder after the toss?",
    "Negation / constraint": "Can a declaration or follow-on decision be withdrawn after it has been notified?",
    "Negation / constraint (2)": "Are unbowled overs carried forward to the next day?",
    "Other (buried at rank 4-10)": "What is the penalty for running short deliberately?",
    "Limitation (retrieval OK, LLM misreads)": "Can a captain declare in the final innings of the match?",
}

# OPTIONAL: tell the notebook where the right answer is, so PART 4 can judge automatically.
#   step 1: find_chunks("final innings")   -> shows which chunks contain that text
#   step 2: add an entry like this and re-run the cell:
# ANSWER_KEY["Can a captain declare in the final innings of the match?"] = {
#     "clause_text": "final innings",      # text found in the right chunk
#     "answer_words": ["not"],             # words a correct answer must contain
# }

failure_results = {}
for label, q in FAILURE_QUERIES.items():
    print("\n\n>>>>>>>>>> CATEGORY:", label, "<<<<<<<<<<")
    failure_results[label] = run_both(q)
    report(failure_results[label])



>>>>>>>>>> CATEGORY: Keyword trap / distractor <<<<<<<<<<
QUERY: How much extra time can be used to complete the minimum overs?

PART 1 - RANKING OF THE TOP 10 CHUNKS
  Rank_before = position given by the bi-encoder (cosine similarity)
  Rank_after  = position after the cross-encoder re-sorts the same 10 chunks
  Change      = places moved (+ means moved up, - means moved down)
  Cosine      = bi-encoder similarity | CE_score = cross-encoder score (higher = more relevant)

 Rank_after  Rank_before Change  Chunk  Page  Cosine  CE_score                           Note
          1            2     +1     74    25   0.708      1.90                               
          2           10     +8     68    23   0.627      1.50 ENTERED top 3  <- right clause
          3            3      0     67    23   0.685      1.24                <- right clause
          4            5     +1     73    25   0.665      1.05                               
          5            8     +3     71    24   0.6

### Failure query write-up (fill in after running)

Count a case as a real failure only when **Answer 1 is wrong or incomplete** and Answer 2 fixes it. Where the answers match despite a rank shift, note that as a limitation (re-ranking helps retrieval ordering but does not always change the final answer).


In [47]:
# ---------- one summary block per failure query ----------
summary_rows = []
for label, res in failure_results.items():
    entered, left = entered_and_left(res)
    q = res["query"]
    summary_rows.append(
        {
            "Category": label,
            "Query": q,
            "Entered_top3": ", ".join(
                f"chunk {d['id']} (rank {d['orig_rank']} -> {d['new_rank']})"
                for d in entered
            )
            or "none",
            "Left_top3": ", ".join(
                f"chunk {d['id']} (rank {d['orig_rank']} -> {d['new_rank']})"
                for d in left
            )
            or "none",
            "Answer1_has_expected_words": has_words(res["ans1"], q),
            "Answer2_has_expected_words": has_words(res["ans2"], q),
            "Answer_similarity_pct": similarity(res["ans1"], res["ans2"]),
            "Result": result_sentence(res),
        }
    )
summary = pd.DataFrame(summary_rows)

print("SUMMARY OF ALL FAILURE QUERIES")
for row in summary_rows:
    print("\n" + "-" * 90)
    for key, value in row.items():
        print(f"{key:28}: {value}")

print(
    "\n\nRANK-SHIFT SENTENCES FOR THE REPORT (check each one against the printed chunk text):"
)
for label, res in failure_results.items():
    entered, left = entered_and_left(res)
    for d in entered:
        print(
            f"- [{label}] Chunk {d['id']} moved from Rank #{d['orig_rank']} to Rank #{d['new_rank']} for: \"{res['query']}\""
        )
    for d in left:
        print(
            f"- [{label}] Chunk {d['id']} dropped from Rank #{d['orig_rank']} to Rank #{d['new_rank']} for: \"{res['query']}\""
        )

summary.to_csv("failure_summary.csv", index=False)
print("\nSaved a copy of the summary to failure_summary.csv")

SUMMARY OF ALL FAILURE QUERIES

------------------------------------------------------------------------------------------
Category                    : Keyword trap / distractor
Query                       : How much extra time can be used to complete the minimum overs?
Entered_top3                : chunk 68 (rank 10 -> 2)
Left_top3                   : chunk 83 (rank 1 -> 7)
Answer1_has_expected_words  : No
Answer2_has_expected_words  : Yes
Answer_similarity_pct       : 11
Result                      : No difference: both pipelines found the right clause.

------------------------------------------------------------------------------------------
Category                    : Exception / corner case
Query                       : Can a team nominate a substitute fielder after the toss?
Entered_top3                : chunk 3 (rank 4 -> 1), chunk 9 (rank 5 -> 2)
Left_top3                   : chunk 264 (rank 2 -> 4), chunk 147 (rank 3 -> 6)
Answer1_has_expected_words  : No
Answer2_has_expec

### Export the failure queries to CSV
The next cell saves every **failure query** with both answers, the chunks used and the rank changes to **`questions_and_answers.csv`** (opens in Excel, Numbers or Google Sheets). It needs no extra LLM calls because it uses the results from Step 9. The CSV for *all* candidate questions is created in Step 8b.

In [48]:
def result_row(label, res):
    """One CSV row for one query: question, chunks, both answers, rank changes, timings."""
    entered, left = entered_and_left(res)
    q = res["query"]
    return {
        "Category": label,
        "Question": q,
        "Baseline_top3_chunks": ", ".join(str(c) for c in res["base_top3"]),
        "Reranked_top3_chunks": ", ".join(str(d["id"]) for d in res["rr_top3"]),
        "Entered_top3 (chunk: rank before -> after)": ", ".join(
            f"{d['id']}: {d['orig_rank']}->{d['new_rank']}" for d in entered
        )
        or "none",
        "Left_top3 (chunk: rank before -> after)": ", ".join(
            f"{d['id']}: {d['orig_rank']}->{d['new_rank']}" for d in left
        )
        or "none",
        "Answer_1_baseline": res["ans1"],
        "Answer_2_reranked": res["ans2"],
        "Answer_similarity_pct": similarity(res["ans1"], res["ans2"]),
        "Result": result_sentence(res),
        "Retrieval_ms": round(res["t_baseline"] * 1000, 1),
        "Rerank_extra_ms": round(res["t_rerank"] * 1000, 1),
        "LLM_answer1_s": round(res["t_llm1"], 2),
        "LLM_answer2_s": round(res["t_llm2"], 2),
    }


export_df = pd.DataFrame(
    [result_row(label, res) for label, res in failure_results.items()]
)
export_df.to_csv(
    "questions_and_answers.csv", index=False, encoding="utf-8-sig"
)  # utf-8-sig so Excel shows characters correctly
print(f"Saved {len(export_df)} rows to questions_and_answers.csv\n")
print(
    export_df[["Category", "Question", "Answer_similarity_pct"]].to_string(index=False)
)

Saved 6 rows to questions_and_answers.csv

                               Category                                                                         Question  Answer_similarity_pct
              Keyword trap / distractor                   How much extra time can be used to complete the minimum overs?                     11
                Exception / corner case                         Can a team nominate a substitute fielder after the toss?                     20
                  Negation / constraint Can a declaration or follow-on decision be withdrawn after it has been notified?                      9
              Negation / constraint (2)                              Are unbowled overs carried forward to the next day?                     40
            Other (buried at rank 4-10)                              What is the penalty for running short deliberately?                     13
Limitation (retrieval OK, LLM misreads)                         Can a captain declare in the 

## Step 10: Latency profiling

Table 1 times only the retrieval stages (repeated runs, warm-up excluded). Table 2 shows the time of each query from start to finish, including the LLM.

In [49]:
def profile(queries, repeats=5):
    """Time the two retrieval stages many times so the numbers are stable."""
    first = baseline_retrieve(queries[0])
    rerank(queries[0], first)  # warm-up run (not counted)
    bi_times, ce_times = [], []
    for q in queries:
        for _ in range(repeats):
            t0 = time.perf_counter()
            cands = baseline_retrieve(q)
            t1 = time.perf_counter()
            rerank(q, cands)
            t2 = time.perf_counter()
            bi_times.append((t1 - t0) * 1000)
            ce_times.append((t2 - t1) * 1000)
    return np.array(bi_times), np.array(ce_times)


prof_queries = list(FAILURE_QUERIES.values())
bi_ms, ce_ms = profile(prof_queries)


def describe(x):
    return {
        "Mean_ms": x.mean(),
        "Median_ms": np.median(x),
        "Std_ms": x.std(),
        "Min_ms": x.min(),
        "Max_ms": x.max(),
    }


lat = pd.DataFrame(
    {
        "Pipeline 1: bi-encoder search": describe(bi_ms),
        "Pipeline 2 extra: cross-encoder on 10 chunks": describe(ce_ms),
        "Pipeline 2 total (search + re-rank)": describe(bi_ms + ce_ms),
    }
).T.round(2)

print(
    f"TABLE 1 - Retrieval time ({len(prof_queries)} queries x 5 repeats, warm-up excluded)\n"
)
print(lat.to_string())
print(
    f"\nRe-ranking adds about {ce_ms.mean():.0f} ms per query (about {ce_ms.mean()/10:.1f} ms per chunk)."
)
print(
    f"That makes the retrieval stage about {(bi_ms + ce_ms).mean() / bi_ms.mean():.0f} times slower than baseline retrieval alone."
)
lat.to_csv("latency_retrieval.csv")

TABLE 1 - Retrieval time (6 queries x 5 repeats, warm-up excluded)

                                              Mean_ms  Median_ms  Std_ms  Min_ms  Max_ms
Pipeline 1: bi-encoder search                    5.91       5.76    0.67    5.23    8.52
Pipeline 2 extra: cross-encoder on 10 chunks    51.54      50.52    5.35   42.02   60.52
Pipeline 2 total (search + re-rank)             57.46      56.01    5.50   47.55   67.17

Re-ranking adds about 52 ms per query (about 5.2 ms per chunk).
That makes the retrieval stage about 10 times slower than baseline retrieval alone.


In [50]:
rows = []
for label, res in failure_results.items():
    rows.append(
        {
            "Query": label,
            "P1_retrieval_ms": res["t_baseline"] * 1000,
            "P2_retrieval_ms": (res["t_baseline"] + res["t_rerank"]) * 1000,
            "Extra_rerank_ms": res["t_rerank"] * 1000,
            "P1_LLM_s": res["t_llm1"],
            "P2_LLM_s": res["t_llm2"],
            "P1_total_s": res["t_baseline"] + res["t_llm1"],
            "P2_total_s": res["t_baseline"] + res["t_rerank"] + res["t_llm2"],
        }
    )
e2e = pd.DataFrame(rows)
average = {"Query": "AVERAGE"}
for col in e2e.columns[1:]:
    average[col] = e2e[col].mean()
e2e = pd.concat([e2e, pd.DataFrame([average])], ignore_index=True).round(2)

print(
    "TABLE 2 - Time for each query, start to finish (P1 = baseline, P2 = with re-ranking)"
)
print(
    "(the first query may be a little slower because the models are still warming up)\n"
)
print(e2e.to_string(index=False))

avg = e2e.iloc[-1]
share = 100 * avg["Extra_rerank_ms"] / 1000 / max(avg["P2_total_s"], 1e-9)
print(
    f"\nOn average re-ranking adds {avg['Extra_rerank_ms']:.0f} ms, which is {share:.1f}% of the total time of Pipeline 2."
)
print(
    f"Each LLM answer takes about {avg['P1_LLM_s']:.1f} s, so the LLM is the slow part of both pipelines."
    if avg["P1_LLM_s"] * 1000 > avg["Extra_rerank_ms"]
    else "Re-ranking is a noticeable part of the total time here."
)
e2e.to_csv("latency_per_query.csv", index=False)

TABLE 2 - Time for each query, start to finish (P1 = baseline, P2 = with re-ranking)
(the first query may be a little slower because the models are still warming up)

                                  Query  P1_retrieval_ms  P2_retrieval_ms  Extra_rerank_ms  P1_LLM_s  P2_LLM_s  P1_total_s  P2_total_s
              Keyword trap / distractor            96.95           150.62            53.67      5.13      1.20        5.22        1.35
                Exception / corner case            53.94           117.27            63.32      7.39      9.31        7.44        9.42
                  Negation / constraint            68.84           128.71            59.87      2.91      3.04        2.98        3.17
              Negation / constraint (2)            47.20            94.41            47.22      7.79      5.60        7.84        5.70
            Other (buried at rank 4-10)            62.66           122.82            60.16      4.54      8.32        4.60        8.44
Limitation (retrieval O